In [ ]:
from pathlib import Path
from IPython.display import Audio

In [ ]:
from sj_utils.file.yaml import load_yaml
from sj_utils.collection import SafetyDict
from sj_utils.audio import segment_audio, load_audio_from_mp4
from sj_ai_utils.datasets.l_hotse import AMI

In [ ]:
from rt_whisper import streamers, saveloaders
from rt_whisper.data import Param, TokenState

In [ ]:
SEED = 42
SAMPLE_RATE = 16000
PATH = "/workspaces/dev/.datasets/ami"
HYPERPARAMETER = "/workspaces/dev/test/optimize/esic/hyperparameters/20250811/step1_16b-96k/trial_wer4o3_4380_20250812_052633.yaml"
LOG_FILE_PATH = "/workspaces/dev/logs/RTWhisper.log"

In [ ]:
path = Path(PATH)
log = Path(LOG_FILE_PATH)
if not path.exists():
    raise FileNotFoundError(f"Path {path} does not exist.")
if log.exists():
    with log.open("w"): pass
hyperparameter = Path(HYPERPARAMETER)

In [ ]:
dataset = AMI(path).load_dev_ihm().sample(1)

In [ ]:
for _, audio, _ in dataset:
    # audio, sr = librosa.load(src, sr=SAMPLE_RATE)
    segments = segment_audio(audio)
Audio(audio, rate=16000)

In [ ]:
hyperparameter = SafetyDict(load_yaml(hyperparameter)[1])

In [ ]:
# SAVED_PATH = "/workspaces/dev/storage/esic/112000/dev/20090203/014_017_EN_Ždanoka/en.OS.man-diar"
# saved_path = Path(SAVED_PATH)
# token_streamer = saveloaders.get_token_streamer_loader(saved_path, hyperparameter)

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2_min_filter(hyperparameter=hyperparameter)
# token_streamer = streamers.get_token_streamer_with_vad_v2_min_filter()

In [ ]:
param = Param()
completed = []
index = -1

In [ ]:
from rt_whisper.processors.asr.data import ASRState

index += 1
chunk = segments[index]

param.chunk = chunk
param.language = "en"
ctx:TokenState = token_streamer.process(param, get_context = True)
result = ctx.extract()
completed.extend(result.completed)
param.update(result, update_prompt=True)

print(f"{index}" + "--" * 20)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.completed]
)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.candidate]
)

In [ ]:
Audio(result.context_dict[ASRState].chunk, rate=sr)

In [ ]:
param = Param()
completed = []
for i, segment in enumerate(segments):
    param.chunk = segment
    param.language = "en"
    ctx:TokenState = token_streamer.process(param, get_context = True)
    result = ctx.extract()
    completed.extend(result.completed)
    param.update(result, update_prompt=True)

completed.extend(result.candidate)

In [ ]:
for s in completed:
    print(s.lang, s.text)